# Preprocessing Driver

Runs the full preprocessing pipeline to generate model-ready `.mat` files.
Run this notebook whenever you add new subjects or want to regenerate the processed data.

**Steps:**
1. Process each raw per-subject `.mat` file (bandpass → resample → z-score)
2. Combine all processed subject files into a single model-ready file

Output of step 2 is what `model.ipynb` loads for training.

In [ ]:
from preprocessing import batch_process_subjects, combine_processed_files

In [ ]:
# ── Preprocessing toggles ─────────────────────────────────────────────────────
APPLY_BANDPASS = True    # Apply 20–500 Hz bandpass filter before resampling
APPLY_ZSCORE   = True    # Apply per-channel z-score normalization after resampling
TARGET_LENGTH  = 1500    # Number of samples after resampling

# ── Paths ─────────────────────────────────────────────────────────────────────
# Raw per-subject .mat files from the MATLAB pipeline
INPUT_DIR    = '/Users/chrisdollo/Documents/Research/putEMG prime/Data/X/gesture_per_subject_data_5'

# Processed per-subject output directory
OUTPUT_DIR   = '/Users/chrisdollo/Documents/Research/putEMG prime/Data/X/preprocessed_file_5'

# Final combined file loaded by model.ipynb
COMBINED_OUT = '/Users/chrisdollo/Documents/Research/putEMG prime/Data/X/model_ready_5/model_ready_5_sub.mat'

print(f'Bandpass filter : {"ON" if APPLY_BANDPASS else "OFF"}')
print(f'Z-score norm    : {"ON" if APPLY_ZSCORE else "OFF"}')
print(f'Resample target : {TARGET_LENGTH} samples')

In [ ]:
# Step 1: Process each subject file
batch_process_subjects(
    input_dir      = INPUT_DIR,
    output_dir     = OUTPUT_DIR,
    apply_bandpass = APPLY_BANDPASS,
    apply_zscore   = APPLY_ZSCORE,
    target_length  = TARGET_LENGTH,
)

# Step 2: Combine all processed files into one
combine_processed_files(
    input_dir   = OUTPUT_DIR,
    output_path = COMBINED_OUT,
)

---
## Optional — Inspect a Single Gesture Before/After Preprocessing

In [ ]:
import glob
import scipy.io
from preprocessing import (
    bandpass_filter, resample_gesture, zscore_normalize,
    plot_signal_modification, ORIGINAL_FS, TARGET_LENGTH, MAT_KEY
)

sample_files = sorted(glob.glob(INPUT_DIR + '/*.mat'))
if sample_files:
    raw_mat  = scipy.io.loadmat(sample_files[0])[MAT_KEY]
    raw_cell = raw_mat[0, 0]   # first repetition, first gesture

    filtered  = bandpass_filter(raw_cell.astype(float)) if APPLY_BANDPASS else raw_cell
    resampled = resample_gesture(filtered, TARGET_LENGTH)
    normed    = zscore_normalize(resampled) if APPLY_ZSCORE else resampled

    if APPLY_BANDPASS:
        plot_signal_modification(raw_cell, filtered, channel=0,
                                 title='Raw vs Bandpass Filtered')

    plot_signal_modification(raw_cell, normed, channel=0,
                             fs_orig=ORIGINAL_FS,
                             fs_new=ORIGINAL_FS * TARGET_LENGTH / raw_cell.shape[0],
                             title='Raw vs Fully Preprocessed')
else:
    print('No files found in INPUT_DIR — check your path in the config cell.')